In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

df = pd.read_csv('../data/python_qa_clean.csv')
print(df.shape)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\prems\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\prems\AppData\Roaming\nltk_data...


(171, 4)


In [2]:
# Normally stop words, but meaningful in Python, so we KEEP them
KEEP = {'is', 'in', 'not', 'and', 'or', 'if', 'else', 'for', 'while',
        'as', 'with', 'from', 'no', 'all', 'any'}
STOP = set(stopwords.words('english')) - KEEP

# Symbols converted to words so they are not lost (order matters)
SYMBOLS = {
    '**kwargs': ' kwargs ',
    '*args': ' args ',
    '==': ' doubleequals ',
    '!=': ' notequals ',
    ':=': ' walrus ',
    '//': ' floordivision ',
    '**': ' doublestar ',
    '/': ' truedivision ',
    '%': ' modulus ',
}

lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = str(text).lower()
    for symbol, word in SYMBOLS.items():
        text = text.replace(symbol, word)
    tokens = re.findall(r'[a-z0-9_]+', text)
    tokens = [t.strip('_') for t in tokens]          # __init__ -> init
    tokens = [t for t in tokens if t and t not in STOP]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]  # lists -> list
    return ' '.join(tokens)

In [3]:
tests = [
    "What is the difference between == and is?",
    "What does the // operator do?",
    "What are *args and **kwargs?",
    "What is the __init__ method?",
    "How do I reverse a string?",
]
for t in tests:
    print(t, '->', preprocess(t))

What is the difference between == and is? -> is difference doubleequals and is
What does the // operator do? -> floordivision operator
What are *args and **kwargs? -> args and kwargs
What is the __init__ method? -> is init method
How do I reverse a string? -> reverse string


In [4]:
df['clean_question'] = df['question'].apply(preprocess)

print("Empty after cleaning:", (df['clean_question'].str.strip() == '').sum())

df.to_csv('../data/python_qa_processed.csv', index=False)
df[['question', 'clean_question']].head(10)

Empty after cleaning: 0


,question,clean_question
0,What is Python?,is python
1,Who created Python?,created python
2,What are the features of Python?,feature python
3,Is Python compiled or interpreted?,is python compiled or interpreted
4,What is PEP 8?,is pep 8
5,What are variables in Python?,variable in python
6,What are the rules for naming variables in Pyt...,rule for naming variable in python
7,What are keywords in Python?,keywords in python
8,What are the built-in data types in Python?,built in data type in python
9,What is dynamic typing in Python?,is dynamic typing in python
